In [7]:
!pip install sacremoses

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 867.8/867.8 kB 14.7 MB/s eta 0:00:00


In [1]:
import numpy as np
import pandas as pd
import torch
import re
from sklearn.cluster import DBSCAN
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForSeq2SeqLM
from huggingface_hub import notebook_login, hf_hub_download
from tqdm.auto import tqdm

Для вычислений используется cpu


In [3]:
notebook_login()

In [ ]:
#Загрузка исходных данных из Kaggle
import kagglehub

kagglehub.login()
path_file = kagglehub.competition_download('llm-classification-finetuning',
                                           path='train.csv')
origin_data = pd.read_csv(path_file)

In [4]:
#Загрузка исходных данных из локального файла
origin_data = pd.read_csv('train.csv')

# **Методы**

In [5]:
#Генерация текста моделью
def generate(model,
             model_params,
             tokenizer,
             texts,
             batch_size,
             device):
    if not isinstance(texts, list):
        raise TypeError('Ожидался тип List')
    
    model.to(device)
    model.eval()
    with torch.no_grad():
        result = []
        for index in tqdm(range(0, len(texts), batch_size)):
            batch = texts[index:index+batch_size]
            inputs = tokenizer(batch,
                               padding=True,
                               return_tensors='pt')
            inputs = {param: value.to(device) for param, value in inputs.items()}
            outputs = model.generate(**inputs, **model_params).cpu()
            generated_text = tokenizer.decode(outputs, skip_special_tokens=True)
            del inputs, outputs
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            result.extend(generated_text)
    return result

In [6]:
#Формирует название семейства LLM из названия конкретной LLM
def getLLMFamily(dataset):
    def formFamilyName(words):
        name = words[0]
        if name == 'gpt4all':
            return 'gpt4'
        elif name.find('qwen') != -1:
            return 'qwen'
        elif name in ('falcon', 'nous', 'dolphin'):
            return 'other'
        else:
            if words[1].isdigit():
                name += words[1]
            return name
        
    if not isinstance(dataset, pd.Series):
        raise TypeError('Ожидался тип pd.Series')
    return dataset.str.split(r'-|\.').apply(formFamilyName)

In [7]:
#Подготовка данных для парафраза
def findSeqEnd(text, separators, threshold):
    """
    Поиск позиции последнего вхождения разделителя из separators по приоритету в часте текста text[:threshold].
    Индекс разделителя в separators задает его приоритет
    Параметры
    --------
    text: str
        Входящий текст

    separators: List                              
        Приоритетный список разделителей

    threshold: int
        Ограничение по длине обрабатываемого текста
    """
    if not isinstance(separators, list):
        raise TypeError('Ожидался тип List для separators')
        
    separators_end = {char: index for index, char in enumerate(text[:threshold]) 
                           if char in separators}
    for sep in separators:
        if separators_end.get(sep, 0) > 0:
            return separators_end[sep]
    return threshold

def cutText(text, separators, threshold):
    """
    Сжатие текста до длины <= threshold с разделителем " ... " между частями. Если длина текста и так <= threshold, то 
    возвращается исходный текст
    Параметры
    --------
    text: str
        Входящий текст

    separators: List                              
        Приоритетный список разделителей

    threshold: int
        Ограничение по длине выводимого текста
    """
    if len(text) <= threshold:
        return text
    else:
        threshold -= 5
        begin = findSeqEnd(text, separators, threshold // 2)
        end = findSeqEnd(text[::-1], separators, threshold // 2)
        return text[:begin] + ' ... ' + text[-end:]    

def getParaphraseData(dataset, amount_interval, threshold):
    class_amount = dataset['model_family'].value_counts()
    classes = class_amount[class_amount.between(*amount_interval)].index
    data = dataset[dataset['model_family'].isin(classes)]
    seps = ['.', ' ']
    data['response'] = 'paraphrase: ' + data['response'].apply(lambda x: cutText(x, seps, threshold))
    data['prompt'] = 'paraphrase: ' + data['prompt'].apply(lambda x: cutText(x, seps, threshold))
    return data

#Парафраз текста моделью
def paraphrase(model,
               model_params,
               tokenizer,
               dataset,
               batch_size,
               device):
    if not isinstance(dataset, pd.DataFrame):
        raise TypeError('Ожидался тип pd.DataFrame')
    if 'num_return_sequences' not in model_params:
        raise ValueError('Не задано кол-во возвращаемых вариантов num_return_sequences')

    paraphrased_prompt = generate(model, model_params, tokenizer, dataset['prompt'].to_list(), batch_size, device)
    paraphrased_response = generate(model, model_params, tokenizer, dataset['response'].to_list(), batch_size, device)
    model_family = []
    for name in dataset['model_family']:
        model_family.extend([name] * model_params['num_return_sequences'])
    result = {'prompt': paraphrased_prompt,
              'response': paraphrased_response,
              'model_family': model_family}
    return pd.DataFrame(result)

# **1. Определение типа LLM по запрос-ответ**

В тестовом наборе данных нет поля с типом LLM. Обучим модель классификации, которая по запросу пользователя и ответу LLM 
будет предсказывать тип ответевшей LLM.

## **1-1. Загрузка данных**

Объединим в один датасет все prompt и response, оставим только чистый текст и разделим выборку на тестовую и обучающую.

In [8]:
data = pd.DataFrame(np.vstack((origin_data[['prompt','response_a','model_a']],
                               origin_data[['prompt','response_b','model_b']])),
                    columns=['prompt','response','model'])

data['prompt'] = data['prompt'].str[2:-2]
data['response'] = data['response'].str[2:-2]

data_training, data_testing = train_test_split(data,
                                               test_size=0.2,
                                               random_state=101,
                                               stratify=data['model'])

## **1-2. Балансировка классов**

Для качественной классификации необходимо уменьшить дисбаланс в классах.

### **1-2-1. Объединение классов в семейства LLM**

У нас получилось 40 семейств. Для каждого семейства оставим 2000-2500 примеров.

In [9]:
data_training['model_family'] = getLLMFamily(data_training['model'])
data_training = data_training.drop('model', axis=1)

### **1-2-3. Парафраз для миноритарных классов**

Мы можем увеличить объем миноритарных классов путем перефразирования текстов с помощью специальных моделей, 
которые сохранят исходную стилистику и смысл. Для парафраза (и для будущей классификации) существует ограничение по токенам на входные и выходные последовательности (**512 токенов**). Если длина текста больше **1300 символов**, то мы возьмем только начальную и завершающую часть текста (постараемся брать цельные предложения).

In [91]:
tokenizer_paraphrase = AutoTokenizer.from_pretrained("humarin/chatgpt_paraphraser_on_T5_base")
model_paraphrase = AutoModelForSeq2SeqLM.from_pretrained("humarin/chatgpt_paraphraser_on_T5_base")

Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]

In [28]:
model_params = {'repetition_penalty': 1.2,
                'no_repeat_ngram_size': 3,
                'max_new_tokens': 400}

1. Для классов с мощностью до 700 будем генерировать 4 новых текста.

In [ ]:
#Первый батч за 86.96 секунд
data_for_paraphrase = getParaphraseData(data_training, (0, 700), 1300)
model_params['num_beams'] = 4
model_params['num_return_sequences'] = 4
data_paraphrased_1 = paraphrase(model_paraphrase, 
                                model_params, 
                                tokenizer_paraphrase,
                                data_for_paraphrase.iloc[:3],
                                16,
                                device)

2. Для классов с мощностью от 700 до 1000 будем генерировать 2 новых текста.

In [ ]:
data_for_paraphrase = getParaphraseData(data_training, (700, 1000), 1300)
model_params['num_beams'] = 4
model_params['num_return_sequences'] = 2
data_paraphrased_2 = paraphrase(model_paraphrase, 
                                model_params, 
                                tokenizer_paraphrase,
                                data_for_paraphrase,
                                16,
                                device)

3. Для классов с мощностью от 1000 до 1700 будем генерировать 1 новый текст.

In [ ]:
data_for_paraphrase = getParaphraseData(data_training, (1000, 1700), 1300)
model_params['num_beams'] = 4
model_params['num_return_sequences'] = 1
data_paraphrased_3 = paraphrase(model_paraphrase, 
                                model_params, 
                                tokenizer_paraphrase,
                                data_for_paraphrase,
                                16,
                                device)

In [ ]:
data_training = pd.concat([data_training,
                           data_paraphrased_1,
                           data_paraphrased_2,
                           data_paraphrased_3],
                          ignore_index=True)

### **1-2-4. Уменьшение мажоритарных классов**

Для уменьшения мажоритарных классов можно использовать случайное удаление строк. Оставим не более 
**3000** примеров.

In [83]:
data_training['group_field'] = data_training['model_family']
data_training = data_training.groupby('group_field', 
                                      group_keys=False).apply(lambda x: x.sample(min(x.shape[0], 3000),
                                                                                 random_state=101))

## **1-3. Обучение модели классификации**